In [5]:
import numpy as np
import pandas as pd
import os


In [7]:
df = pd.read_csv("/home/serdic/project/AID/outputs/01_intrusion/20260223_014008/state_factors_img640_fps10_conf030.csv")

print(df.shape)
print(df.dtypes)

print(df["y_state"].value_counts().sort_index())

print(df.groupby("y_state")[["f_dist","f_ov","p_gap","rep_conf"]].agg(["mean","std"]))

# 영상별 row 수(너무 적은 비디오 찾기)
print(df.groupby("video_id").size().sort_values().head(10))

(9187, 21)
video_id              object
event_idx              int64
clip_path             object
fps                  float64
clip_start_sec       float64
t_sec                float64
orig_frame             int64
y_state                int64
f_dist               float64
f_ov                 float64
p_gap                float64
rep_bbox_x1          float64
rep_bbox_y1          float64
rep_bbox_x2          float64
rep_bbox_y2          float64
rep_conf             float64
rep_policy            object
event_policy          object
cand_window_sec      float64
hold_det                bool
drop_other_events       bool
dtype: object
y_state
0    4711
1    1208
2    3268
Name: count, dtype: int64
           f_dist                f_ov               p_gap            rep_conf  \
             mean       std      mean       std      mean       std      mean   
y_state                                                                         
0        0.155392  0.324312  0.171483  0.371429  0.815599  

# state_factors.csv 컬럼 설명

> 목적: 프레임(샘플링된 update frame)마다 **대표 1인**을 선택해 factor(3개)와 라벨을 저장한 학습/EDA용 테이블

---

## 기본 식별자/소스

- **video_id** (str)  
  비디오 ID. 예: `E01_007`

- **event_idx** (int)  
  해당 비디오에서 선택된 이벤트 인덱스.  
  현재 파이프라인(`event_policy=first`)에서는 대부분 `0`.

- **clip_path** (str)  
  사용한 클립 파일 경로. 예: `data/clips/<VIDEO_ID>/ev00_..._50s.mp4`

---

## 시간/프레임 메타

- **fps** (float)  
  원본(라벨 좌표계) FPS. 보통 `30.0`

- **clip_start_sec** (float)  
  원본 비디오 기준으로 클립이 시작되는 시각(초).  
  `orig_frame` 계산에 사용됨.

- **t_sec** (float)  
  클립 내부에서의 시간(초). (샘플링된 프레임의 timestamp)

- **orig_frame** (int)  
  원본 비디오 좌표계의 프레임 인덱스(라벨 `event_frame`과 같은 좌표계).  
  계산: `orig_frame ≈ round((t_sec + clip_start_sec) * fps)`

---

## 라벨(정답 상태)

- **y_state** (int)  
  정답 상태 라벨.
  - `0` = OUT (이벤트 구간 밖)
  - `1` = CAND (이벤트 시작/끝 전후 cand_window 구간)
  - `2` = IN  (이벤트 구간 내부: `event_frame`)

---

## Factor(학습 feature 핵심 3개)

- **f_dist** (float, 보통 0~1)  
  ROI 경계 기준 거리 신호(근접+침투 기반).  
  - ROI 안쪽(침투)일수록 ↑  
  - ROI 밖에서도 경계 근처면 ↑

- **f_ov** (float, 0~1)  
  bbox 하단 band와 ROI의 overlap 비율(침입 근거 신호).  
  - ROI 밟을수록 ↑

- **p_gap** (float, 보통 0~1)  
  y-guard(“떠있음”) 패널티 신호.  
  - bbox 바닥이 ROI 바닥선 대비 떠있으면 패널티 ↑  
  - 바닥에 붙어있으면 패널티 ↓  
  *(구현 정의에 따라 방향이 다를 수 있으니, 현재 구현 기준 해석을 고정해두는 게 좋음)*

---

## 대표 bbox 정보(디버깅/근거용)

- **rep_bbox_x1, rep_bbox_y1, rep_bbox_x2, rep_bbox_y2** (float)  
  대표 인원의 bbox 좌표(xyxy). (클립 프레임 좌표계)

- **rep_conf** (float)  
  대표 bbox의 detection confidence.

---

## 정책/옵션 메타(실험 재현용)

- **rep_policy** (str)  
  대표 인원 선택 정책. 예: `max_f_ov`

- **event_policy** (str)  
  비디오에서 이벤트 1개 선택 정책. 예: `first` 또는 `longest`

- **cand_window_sec** (float)  
  CAND 라벨링에 쓰는 경계 전후 윈도우(초). 예: `3.0`

- **hold_det** (bool)  
  non-update 프레임에서 det를 유지(hold)하는지 여부.  
  *(현재 CSV는 update frame만 기록되므로 실질 영향은 제한적)*

- **drop_other_events** (bool)  
  동일 비디오 내 다른 이벤트 구간에 해당하는 프레임을 데이터에서 제거하는지 여부(라벨 오염 방지).

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9187 entries, 0 to 9186
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   video_id           9187 non-null   object 
 1   event_idx          9187 non-null   int64  
 2   clip_path          9187 non-null   object 
 3   fps                9187 non-null   float64
 4   clip_start_sec     9187 non-null   float64
 5   t_sec              9187 non-null   float64
 6   orig_frame         9187 non-null   int64  
 7   y_state            9187 non-null   int64  
 8   f_dist             9187 non-null   float64
 9   f_ov               9187 non-null   float64
 10  p_gap              9187 non-null   float64
 11  rep_bbox_x1        9187 non-null   float64
 12  rep_bbox_y1        9187 non-null   float64
 13  rep_bbox_x2        9187 non-null   float64
 14  rep_bbox_y2        9187 non-null   float64
 15  rep_conf           9187 non-null   float64
 16  rep_policy         9187 